In [236]:
import os
os.chdir("/Users/naginilandarhi/DSAICourse/Prof_Shankar_Venkatagiri/LinAlg4DataScience/faker_data_synthesizer")

In [237]:
import pandas as pd
import numpy as np
import time as time
import re
from pathlib import Path

In [238]:
print(os.getcwd())

/Users/naginilandarhi/DSAICourse/Prof_Shankar_Venkatagiri/LinAlg4DataScience/faker_data_synthesizer


I need to do is to deal with id column. I will drop the id column from the data matrix. I will keep only the feature columns in the data matrix.

Now I need to identify which cluster each row belongs to. I do not want to use KMeans from sklearn. I want to implement KMeans from scratch. I will use the following steps:
1. Randomly select k rows from the data matrix as the initial centroids.
2. Assign each row to the nearest centroid based on Euclidean distance.
3. Recalculate the centroids as the mean of the rows assigned to each centroid.
4. Repeat steps 2 and 3 until convergence (i.e., the centroids do not change significantly between iterations). 

In [239]:
def kmeans_cpu(X,
               K,
               initial_centroids=None,
               max_iters=100,
               tol=1e-4,
               seed=42):
    N, D = X.shape
    if initial_centroids is None:
        rng = np.random.default_rng(seed)
        indices = rng.choice(N, K, replace=False)
        centroids = X[indices].copy()

    else:
        centroids = initial_centroids.copy()
        centroids = centroids.astype(np.float32)

    assignments = np.full(N, -1, dtype=np.int32)   
    start_time = time.perf_counter()

    for iteration in range(max_iters):
        distances = np.linalg.norm(X[:, np.newaxis, :] - centroids[np.newaxis, :, :], axis=2)
        new_assignments = np.argmin(distances, axis=1)

        if np.array_equal(assignments, new_assignments):
            print(f"Converged after {iteration} iterations.")
            break

        assignments = new_assignments
        new_centroids = np.empty_like(centroids)

        for k in range(K):
            assigned_points = X[assignments == k]
            if len(assigned_points) > 0:
                new_centroids[k] = assigned_points.mean(axis=0)
                new_centroids[k] = new_centroids[k].astype(np.float32)

        if np.array_equal(centroids, new_centroids):
            print(f"Converged after {iteration} iterations.")
            break

        movement = np.linalg.norm(
            new_centroids - centroids,
            axis=1
        )

        max_movement = np.max(movement)
        centroids = new_centroids
        if max_movement < tol:
            break

    end_time = time.perf_counter()
    execution_time = end_time - start_time
    return {
        "assignments": assignments,
            "centroids": centroids,
            "execution_time": execution_time,
            "iterations": iteration + 1,
            "max_movement": max_movement
        }        


In [240]:
def representative_points(
    X,
    ids,
    centroids,
    assignments
):
    representatives = []

    for k in range(len(centroids)):

        indices = np.where(assignments == k)[0]

        cluster_points = X[indices]

        diff = cluster_points - centroids[k]

        distances = np.sum(
            diff * diff,
            axis=1
        )

        nearest_local_index = np.argmin(distances)

        original_index = indices[nearest_local_index]

        representatives.append({
            "cluster": k,
            "ID": ids[original_index],
            "distance_to_centroid":
                float(np.sqrt(distances[nearest_local_index]))
        })

    return representatives

In [241]:
base = Path(".")  

row_dirs = sorted(
    (d for d in base.iterdir() if d.is_dir() and re.fullmatch(r"rows\d+", d.name)),
    key=lambda d: int(d.name[4:]),
)


In [242]:
for d in row_dirs:
    n_rows = int(d.name[4:])
    for csv_path in sorted(d.glob("*.csv")):
        df = pd.read_csv(csv_path)
        #print(f"{d.name}/{csv_path.name}: shape={df.shape}")
        data_matrix_a = df.drop(columns=['ID']).to_numpy(dtype=np.float32)
        ids = df["ID"].astype(str).to_numpy()
        rng = np.random.default_rng(42)
        initial_centroids = rng.choice(len(data_matrix_a), size=17, replace=False)
        initial_centroids = data_matrix_a[initial_centroids]
        np.savetxt(
            f"initial_centroids_{csv_path.stem}.csv",
            initial_centroids,
            delimiter=","
            )
        #print("Initial centroids:\n", initial_centroids)
        result = kmeans_cpu(data_matrix_a, K=17, initial_centroids=initial_centroids, max_iters=1000, tol=1e-4, seed=42)
        print(f"Execution time: {result['execution_time']:.4f} seconds")
        print(f"Iterations: {result['iterations']}")
        print(f"Max movement: {result['max_movement']:.6f}")
        print(f"Centroids:\n{result['centroids']}")
        pd.DataFrame({
            "Execution time": [result["execution_time"]],
            "Iterations": [result["iterations"]],
            "Max movement": [result["max_movement"]],
            "Centroids": [result["centroids"].tolist()],
        }).to_csv(f"final_results_{csv_path.stem}.csv", index=False)
        output = pd.DataFrame({
            "ID": ids,
            "Cluster": result["assignments"]
        })
        output.to_csv(f"kmeans_output_{csv_path.stem}.csv", index=False)
        representatives = representative_points(
            data_matrix_a,
            ids,
            result["centroids"],
            result["assignments"]
        )

        pd.DataFrame({
            "Cluster": [rep["cluster"] for rep in representatives],
            "ID": [rep["ID"] for rep in representatives],
            "Distance to Centroid": [rep["distance_to_centroid"] for rep in representatives]
        }).to_csv(f"kmeans_representatives_{csv_path.stem}.csv", index=False)

Converged after 96 iterations.
Execution time: 0.1713 seconds
Iterations: 97
Max movement: 0.020040
Centroids:
[[-77.06335   144.06837   448.3926    -72.57098   114.28932   615.78534
  498.17685   139.43596   544.85095  ]
 [792.34955   -93.81259   788.9139    670.0607    269.7207     67.30047
  954.01025   273.87396     1.0397642]
 [602.1523    -72.81472   203.3655    146.38196   708.7554    645.2615
  881.7233     -4.238897  363.88226  ]
 [-73.39391   136.92284   459.8984    -67.1066    124.358215  610.70966
  508.5554    139.42662   550.92255  ]
 [-65.28345   135.21806   457.20215   -63.18475   117.42248   616.7843
  491.4547    137.94507   548.8895   ]
 [-66.945366  134.94925   450.48172   -74.95981   124.51279   609.7675
  494.7115    148.37766   549.09607  ]
 [794.2857    -95.30381   786.1454    668.7661    276.07614    75.858574
  956.07855   268.41486     3.2075398]
 [-65.18658   149.063     459.2853    -71.3618    118.0988    607.57513
  493.80368   142.41202   545.62134  ]
 [-